In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms
import torch.nn.functional as F

In [4]:
Batch_size = 20
epochs = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
train_loader = torch.utils.data.DataLoader(
        datasets.MNIST('data', train=True, download=True, 
                       transform=transforms.Compose([
                           transforms.ToTensor(),
                           transforms.Normalize((0.1307,), (0.3081,))
                       ])),
        batch_size=Batch_size, shuffle=True)

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data/MNIST/raw/train-images-idx3-ubyte.gz to data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data/MNIST/raw/train-labels-idx1-ubyte.gz to data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data/MNIST/raw/t10k-images-idx3-ubyte.gz to data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%

Extracting data/MNIST/raw/t10k-labels-idx1-ubyte.gz to data/MNIST/raw



In [10]:
test_loader = torch.utils.data.DataLoader(
        datasets.MNIST('data', train=False, 
                       transform=transforms.Compose([
                           transforms.ToTensor(),
                           transforms.Normalize((0.1307,), (0.3081,))
                       ])),
        batch_size=Batch_size, shuffle=True)

In [16]:
class net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,10,5)
        self.conv2 = nn.Conv2d(10,20,3)
        self.fc1 = nn.Linear(20*10*10,500)
        self.fc2 = nn.Linear(500,10)
    def forward(self,x):
        in_size = x.size(0)
        out = self.conv1(x)
        out = F.relu(out)
        out = F.max_pool2d(out,2,2)
        out = self.conv2(out)
        out = F.relu(out)
        out = out.view(in_size,-1)
        out = self.fc1(out)
        out = F.relu(out)
        out = self.fc2(out)
        out = F.log_softmax(out,dim=1)
        return out

In [19]:
model = net().to(DEVICE)
optimizer = optim.Adam(model.parameters())

In [20]:
def train(model,device,train_loader,optimizer,epoch):
    model.train()
    for batch_idx,(data,target) in enumerate(train_loader):
        data,target = data.to(device),target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output,target)
        loss.backward()
        optimizer.step()
        if (batch_idx+1) %30 ==0:
                print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            

In [21]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item() 
            pred = output.max(1, keepdim=True)[1] 
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))

In [23]:
for epoch in range(1, epochs + 1):
    train(model, DEVICE, train_loader, optimizer, epoch)
    test(model, DEVICE, test_loader)

Train Epoch: 1 [580/60000 (1%)]	Loss: 0.819455
Train Epoch: 1 [1180/60000 (2%)]	Loss: 0.225426
Train Epoch: 1 [1780/60000 (3%)]	Loss: 0.060688
Train Epoch: 1 [2380/60000 (4%)]	Loss: 0.389087
Train Epoch: 1 [2980/60000 (5%)]	Loss: 0.675655
Train Epoch: 1 [3580/60000 (6%)]	Loss: 0.142121
Train Epoch: 1 [4180/60000 (7%)]	Loss: 0.250679
Train Epoch: 1 [4780/60000 (8%)]	Loss: 0.120094
Train Epoch: 1 [5380/60000 (9%)]	Loss: 0.136016
Train Epoch: 1 [5980/60000 (10%)]	Loss: 0.514297
Train Epoch: 1 [6580/60000 (11%)]	Loss: 0.269637
Train Epoch: 1 [7180/60000 (12%)]	Loss: 0.047777
Train Epoch: 1 [7780/60000 (13%)]	Loss: 0.194157
Train Epoch: 1 [8380/60000 (14%)]	Loss: 0.219139
Train Epoch: 1 [8980/60000 (15%)]	Loss: 0.081497
Train Epoch: 1 [9580/60000 (16%)]	Loss: 0.417219
Train Epoch: 1 [10180/60000 (17%)]	Loss: 0.037694
Train Epoch: 1 [10780/60000 (18%)]	Loss: 0.088631
Train Epoch: 1 [11380/60000 (19%)]	Loss: 0.072597
Train Epoch: 1 [11980/60000 (20%)]	Loss: 0.159224
Train Epoch: 1 [12580/6000